# Leads Scraper — Medellín Business Database

Automated data pipeline to extract and enrich business leads from Google Maps API across 22 zones in Medellín.

**Sectors covered:** Health (Dentistry) | Aesthetics (Aesthetic Clinics)

**Stack:** Python · Google Maps Places API · Selenium · pandas · requests · regex · openpyxl

## 1. Dependencies

In [ ]:
# !pip install requests pandas openpyxl selenium webdriver-manager

## 2. Imports & Configuration

In [ ]:
import requests
import pandas as pd
import time
import re
from urllib.parse import urljoin

# Google Maps API Key
API_KEY    = "YOUR_GOOGLE_MAPS_API_KEY"   # <-- replace with your key
ARCHIVO    = "leads_medellin_completo.xlsx"
URL_SEARCH = "https://maps.googleapis.com/maps/api/place/textsearch/json"
URL_DETAILS = "https://maps.googleapis.com/maps/api/place/details/json"

print("Configuration loaded")

## 3. Helper Functions

In [ ]:
EMAIL_FILTERS = ['sentry', 'wix', 'google', 'schema', 'example', 'jquery',
                  'png', 'jpg', 'svg', 'css', '.js', '@2x', 'webpack']

def extract_email(url):
    """Extract email from static websites using requests."""
    if not url or pd.isna(url):
        return ""
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=8)
        emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', response.text)
        emails = [e for e in emails if not any(x in e.lower() for x in EMAIL_FILTERS)]
        if emails:
            return emails[0]
        for slug in ["/contacto", "/contact", "/nosotros", "/about", "/contactenos"]:
            try:
                r = requests.get(urljoin(url, slug), headers=headers, timeout=6)
                if r.status_code == 200:
                    emails = re.findall(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', r.text)
                    emails = [e for e in emails if not any(x in e.lower() for x in EMAIL_FILTERS)]
                    if emails:
                        return emails[0]
            except:
                continue
        return ""
    except:
        return ""

def extract_whatsapp(web, phone):
    """Extract WhatsApp number from wa.me link or Colombian mobile number."""
    if web and not pd.isna(web):
        wa = re.findall(r'wa\.me/(\d+)', str(web))
        if wa:
            return "+" + wa[0]
    if phone and not pd.isna(phone):
        number = re.sub(r'\D', '', str(phone))
        if number.startswith('3') and len(number) == 10:
            return "+57" + number
        if number.startswith('573') and len(number) == 12:
            return "+" + number
    return ""

print("Helper functions defined")

## 4. Core Scraper Function

In [ ]:
def scrape_category(zones_dict, category, subcategory, df_existing=None):
    """
    Extract leads from Google Maps for a given category.
    
    Parameters:
        zones_dict: {search_query: commune_name}
        category: str (e.g. 'Health')
        subcategory: str (e.g. 'Dentistry')
        df_existing: existing DataFrame to avoid duplicates
    
    Returns:
        DataFrame with new leads
    """
    if df_existing is not None and "place_id" in df_existing.columns:
        seen_ids = set(df_existing["place_id"].dropna().tolist())
    else:
        seen_ids = set()

    new_leads = []

    for zone, commune in zones_dict.items():
        print(f"Searching: {zone}")
        params = {"query": zone, "key": API_KEY, "language": "es"}

        while True:
            response = requests.get(URL_SEARCH, params=params).json()

            for place in response.get("results", []):
                place_id = place.get("place_id", "")
                if place_id in seen_ids:
                    continue
                seen_ids.add(place_id)

                details = requests.get(URL_DETAILS, params={
                    "place_id": place_id,
                    "fields": "formatted_phone_number,website",
                    "key": API_KEY
                }).json()

                web      = details.get("result", {}).get("website", "")
                phone    = details.get("result", {}).get("formatted_phone_number", "")
                email    = extract_email(web)
                whatsapp = extract_whatsapp(web, phone)

                new_leads.append({
                    "Name":        place.get("name", ""),
                    "Address":     place.get("formatted_address", ""),
                    "Phone":       phone,
                    "Web":         web,
                    "Rating":      place.get("rating", ""),
                    "Email":       email,
                    "Zone":        zone,
                    "place_id":    place_id,
                    "WhatsApp":    whatsapp,
                    "Commune":     commune,
                    "Category":    category,
                    "Subcategory": subcategory
                })
                print(f"  OK: {place.get('name')}")

            next_page_token = response.get("next_page_token")
            if next_page_token:
                time.sleep(3)
                params = {"pagetoken": next_page_token, "key": API_KEY}
            else:
                break

    return pd.DataFrame(new_leads)

print("Scraper function defined")

## 5. Search Zones — Dentistry

In [ ]:
ZONES_DENTISTRY = {
    "dental centers El Poblado Medellin":   "Commune 14 - El Poblado",
    "dental centers Laureles Medellin":     "Commune 11 - Laureles Estadio",
    "dental centers Envigado":              "Municipality - Envigado",
    "dental centers Bello":                 "Municipality - Bello",
    "dental centers Itagui":                "Municipality - Itagui",
    "dental centers Centro Medellin":       "Commune 10 - La Candelaria",
    "dental centers Sabaneta":              "Municipality - Sabaneta",
    "dental centers Belen Medellin":        "Commune 16 - Belen",
    "dental centers Robledo Medellin":      "Commune 7 - Robledo",
    "dental centers Castilla Medellin":     "Commune 5 - Castilla",
    "dental centers Buenos Aires Medellin": "Commune 9 - Buenos Aires",
    "dental centers Manrique Medellin":     "Commune 3 - Manrique",
    "dental centers Aranjuez Medellin":     "Commune 4 - Aranjuez",
    "dental centers Copacabana Antioquia":  "Municipality - Copacabana",
    "dental centers La Estrella Antioquia": "Municipality - La Estrella",
    "dental centers Caldas Antioquia":      "Municipality - Caldas",
    "dental centers Guayabal Medellin":     "Commune 15 - Guayabal",
    "dental centers Estadio Medellin":      "Commune 11 - Laureles Estadio",
    "dental centers Floresta Medellin":     "Commune 11 - Laureles Estadio",
    "dental centers San Javier Medellin":   "Commune 13 - San Javier",
    "dental centers Popular Medellin":      "Commune 1 - Popular",
    "dental centers Santa Cruz Medellin":   "Commune 2 - Santa Cruz",
}

# df_dentistry = scrape_category(ZONES_DENTISTRY, "Health", "Dentistry")
# print(f"Total dentistry leads: {len(df_dentistry)}")

## 6. Search Zones — Aesthetic Clinics

In [ ]:
ZONES_AESTHETICS = {
    "aesthetic clinics El Poblado Medellin":   "Commune 14 - El Poblado",
    "aesthetic clinics Laureles Medellin":     "Commune 11 - Laureles Estadio",
    "aesthetic clinics Envigado":              "Municipality - Envigado",
    "aesthetic clinics Bello":                 "Municipality - Bello",
    "aesthetic clinics Itagui":                "Municipality - Itagui",
    "aesthetic clinics Centro Medellin":       "Commune 10 - La Candelaria",
    "aesthetic clinics Sabaneta":              "Municipality - Sabaneta",
    "aesthetic clinics Belen Medellin":        "Commune 16 - Belen",
    "aesthetic clinics Robledo Medellin":      "Commune 7 - Robledo",
    "aesthetic clinics Castilla Medellin":     "Commune 5 - Castilla",
    "aesthetic clinics Buenos Aires Medellin": "Commune 9 - Buenos Aires",
    "aesthetic clinics Manrique Medellin":     "Commune 3 - Manrique",
    "aesthetic clinics Aranjuez Medellin":     "Commune 4 - Aranjuez",
    "aesthetic clinics Copacabana Antioquia":  "Municipality - Copacabana",
    "aesthetic clinics La Estrella Antioquia": "Municipality - La Estrella",
    "aesthetic clinics Caldas Antioquia":      "Municipality - Caldas",
    "aesthetic clinics Guayabal Medellin":     "Commune 15 - Guayabal",
    "aesthetic clinics Estadio Medellin":      "Commune 11 - Laureles Estadio",
    "aesthetic clinics Floresta Medellin":     "Commune 11 - Laureles Estadio",
    "aesthetic clinics San Javier Medellin":   "Commune 13 - San Javier",
    "aesthetic clinics Popular Medellin":      "Commune 1 - Popular",
    "aesthetic clinics Santa Cruz Medellin":   "Commune 2 - Santa Cruz",
}

# df_aesthetics = scrape_category(ZONES_AESTHETICS, "Aesthetics", "Aesthetic Clinic", df_existing=df_dentistry)
# print(f"Total aesthetics leads: {len(df_aesthetics)}")

## 7. Consolidate & Save Database

In [ ]:
# Consolidate both categories
# df_final = pd.concat([df_dentistry, df_aesthetics], ignore_index=True)
# df_final.to_excel(ARCHIVO, index=False)
# print(f"Total leads saved: {len(df_final)}")

# Load existing database
df = pd.read_excel(ARCHIVO)
print(f"Total records: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print()
print(df["Subcategory"].value_counts())

## 8. Email Enrichment with Selenium (JavaScript rendering)

In [ ]:
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

PAUSE_MIN = 3
PAUSE_MAX = 7
OUTPUT_FILE = "leads_medellin_updated.xlsx"

def create_driver():
    """Initialize headless Chrome driver."""
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    options.add_argument("--log-level=3")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.set_page_load_timeout(15)
    return driver

def extract_email_selenium(driver, url):
    """Extract email from JS-rendered websites using Selenium."""
    if not url or pd.isna(url):
        return ""
    for slug in ["", "/contacto", "/contact", "/nosotros", "/about", "/contactenos"]:
        try:
            driver.get(urljoin(url, slug) if slug else url)
            time.sleep(2)
            emails = re.findall(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}', driver.page_source)
            emails = [e for e in emails if not any(x in e.lower() for x in EMAIL_FILTERS)]
            if emails:
                return emails[0]
            for link in driver.find_elements("css selector", "a[href^='mailto:']"):
                href = link.get_attribute("href")
                if href:
                    email = href.replace("mailto:", "").split("?")[0].strip()
                    if email and "@" in email:
                        return email
        except:
            continue
    return ""

print("Selenium functions defined")

In [ ]:
df = pd.read_excel(ARCHIVO)

SUBCATEGORY = "Aesthetic Clinic"   # change as needed

no_email = df[
    (df["Subcategory"] == SUBCATEGORY) &
    (df["Email"].isna() | (df["Email"] == ""))
]
print(f"{SUBCATEGORY} without email: {len(no_email)}")

driver = create_driver()
found = 0

try:
    for i, row in no_email.iterrows():
        email = extract_email_selenium(driver, row.get("Web"))
        if email:
            df.at[i, "Email"] = email
            found += 1
            print(f"OK  {row['Name']} --> {email}")
        else:
            print(f"--- {row['Name']}")

        if found > 0 and found % 10 == 0:
            df.to_excel(OUTPUT_FILE, index=False)
            print(f"    Partial save: {found} emails found")

        time.sleep(random.uniform(PAUSE_MIN, PAUSE_MAX))
finally:
    driver.quit()
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"\nSaved to: {OUTPUT_FILE}")

total = df[df["Subcategory"] == SUBCATEGORY]
print(f"Emails {SUBCATEGORY}: {total['Email'].notna().sum()} of {len(total)}")
print(f"New emails this session: {found}")

## 9. Database Summary

In [ ]:
df = pd.read_excel(ARCHIVO)

summary = df.groupby("Subcategory").agg(
    Total=("Name", "count"),
    With_Email=("Email", lambda x: x.notna().sum()),
    With_Web=("Web", lambda x: x.notna().sum()),
    With_WhatsApp=("WhatsApp", lambda x: (x != "").sum()),
).reset_index()

print(summary.to_string(index=False))